In [1]:
from rltesting.torch_rl.buffers import ReplayBuffer
from dreamer import *
import gymnasium as gym
from matplotlib import pyplot as plt
from rltesting.torch_rl.utils import random_sample_single_env

import ale_py

gym.register_envs(ale_py)
env = gym.make('ALE/Breakout-v5', render_mode="rgb_array")
env = gym.wrappers.ResizeObservation(env, (64, 64))

A.L.E: Arcade Learning Environment (version 0.10.2+c9d4b19)
[Powered by Stella]


In [2]:
samples = random_sample_single_env(env, num_steps=5000)

In [3]:
print([samples[i].shape for i in range(len(samples))])

[(5000, 64, 64, 3), (5000,), (5000,), (5000,)]


In [4]:
buffer_shapes = [(64, 64, 3), (), (), ()]
dtypes = [np.uint8, np.float32, np.float32, np.float32]
buffer = ReplayBuffer(buffer_shapes, dtypes, buffer_size=4000)

In [5]:
for i in range(5000):
    # buffer.add_sample_until_episode_terminal([samples[j][i] for j in range(len(samples))].copy(), episode_terminal=samples[-1][i])
    buffer.add_sample([samples[j][i] for j in range(len(samples))].copy())

In [6]:
temp = buffer.sample(32, seq_len=8)
temp[0].shape

(8, 32, 64, 64, 3)

In [7]:
temp = buffer.sample(32, seq_len=1)
temp[0].shape

(32, 64, 64, 3)

In [8]:
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.vec_env import SubprocVecEnv
from rltesting.torch_rl.buffers import PerEnvBuffer

def make_env(gym_id):
    def thunk():
        gym.register_envs(ale_py)
        env = gym.make(gym_id, render_mode="rgb_array")
        env = gym.wrappers.ResizeObservation(env, (64, 64))
        return env
    return thunk

num_envs = 16
env = make_vec_env(make_env('ALE/Breakout-v5'), num_envs, vec_env_cls=SubprocVecEnv, vec_env_kwargs=dict(start_method='spawn'))
buffer = PerEnvBuffer(num_envs, buffer_shapes, dtypes, buffer_size=20000)

A.L.E: Arcade Learning Environment (version 0.10.2+c9d4b19)
[Powered by Stella]
A.L.E: Arcade Learning Environment (version 0.10.2+c9d4b19)
[Powered by Stella]
A.L.E: Arcade Learning Environment (version 0.10.2+c9d4b19)
[Powered by Stella]
A.L.E: Arcade Learning Environment (version 0.10.2+c9d4b19)
[Powered by Stella]
A.L.E: Arcade Learning Environment (version 0.10.2+c9d4b19)
[Powered by Stella]
A.L.E: Arcade Learning Environment (version 0.10.2+c9d4b19)
[Powered by Stella]
A.L.E: Arcade Learning Environment (version 0.10.2+c9d4b19)
[Powered by Stella]
A.L.E: Arcade Learning Environment (version 0.10.2+c9d4b19)
[Powered by Stella]
A.L.E: Arcade Learning Environment (version 0.10.2+c9d4b19)
[Powered by Stella]
A.L.E: Arcade Learning Environment (version 0.10.2+c9d4b19)
[Powered by Stella]
A.L.E: Arcade Learning Environment (version 0.10.2+c9d4b19)
[Powered by Stella]
A.L.E: Arcade Learning Environment (version 0.10.2+c9d4b19)
[Powered by Stella]
A.L.E: Arcade Learning Environment (vers

In [9]:
obs = env.reset()
for _ in range(10000):
    action = np.repeat(env.action_space.sample(), num_envs)
    next_obs, reward, done, info = env.step(action)
    buffer.add_sample_until_episode_terminal([obs, action, reward, done])
    obs = next_obs

In [10]:
buffer.sample(32, 2)

In [11]:
buffer.sample(32, 1)

In [13]:
buffer.buffers[0].total

9976